# LegacyAgent V1 - Day 5 - QLoRA Fine-Tuning (cleaned, fully verified)

Turns out the original run (checkpoint-350) never learned to stop generating, because the training labels never included an EOS token. Cell 10 below fixes that by appending the EOS/end-of-turn token to the labeled text.

My first attempt at the EOS fix (steps 0-250, checkpoint qlora_v1_run2_eosfix) showed zero change in eval_loss across every checkpoint -- identical to the decimal at 50/100/150/200/250. Turned out reloading an adapter via PeftModel.from_pretrained doesn't guarantee requires_grad=True on the LoRA parameters. Fixed in the continuation cell near the bottom, which explicitly re-enables gradients before resuming training.

In [1]:
# DAY 5 - CELL 1 (re-run this first)
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/legacyagent")
MANIFEST_DIR = ROOT / "manifests"
CHECKPOINT_DIR = ROOT / "checkpoints" / "qlora_v1_run2"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

EOSFIX_CHECKPOINT_DIR = ROOT / "checkpoints" / "qlora_v1_run2_eosfix"

assert ROOT.exists()
print("Project root:", ROOT)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("EOS-fix checkpoint dir:", EOSFIX_CHECKPOINT_DIR)

Mounted at /content/drive
Project root: /content/drive/MyDrive/legacyagent
Checkpoint dir: /content/drive/MyDrive/legacyagent/checkpoints/qlora_v1_run2
EOS-fix checkpoint dir: /content/drive/MyDrive/legacyagent/checkpoints/qlora_v1_run2_eosfix


In [2]:
# DAY 5 - CELL 2
# GPU check
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    raise RuntimeError("No GPU detected. Runtime -> Change runtime type -> GPU")

CUDA available: True
GPU: Tesla T4
Memory: 14.56 GB


In [3]:
# DAY 5 — CELL 3 (pinned, peft unpinned to match transformers 5.x)
!pip -q install \
    transformers==5.13.0 \
    accelerate \
    peft \
    bitsandbytes \
    librosa \
    soundfile

import transformers, peft
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.1 MB/s eta 0:00:00
transformers: 5.13.0
peft: 0.20.0


In [4]:
# DAY 5 - CELL 4
# Reload frozen train/validation manifests
import json

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_jsonl(MANIFEST_DIR / "train_manifest.jsonl")
val_rows   = load_jsonl(MANIFEST_DIR / "validation_manifest.jsonl")

print("Train rows (raw):", len(train_rows))
print("Validation rows (raw):", len(val_rows))

Train rows (raw): 3702
Validation rows (raw): 903


In [5]:
# DAY 5 - CELL 5
# Filter training-ineligible and implausible-timestamp segments
#
# Excludes:
#   - needs_alignment == True (already excluded from V1 scope)
#   - words/sec > WPS_CEILING (corrupted/misaligned Oyez timestamps)
#
# Genuine short turns (e.g. "Right." at 0.6s, ~3 w/s) are preserved.

import re

WPS_CEILING = 8.0  # sustained spoken English rarely exceeds ~6 w/s

def word_count(text):
    return len(re.findall(r"\b[\w']+\b", text))

def filter_manifest(rows, name):
    ready = []
    dropped_alignment = 0
    dropped_implausible = 0

    for row in rows:
        if row["needs_alignment"]:
            dropped_alignment += 1
            continue

        wps = word_count(row["text"]) / row["duration"] if row["duration"] > 0 else float("inf")

        if wps > WPS_CEILING:
            dropped_implausible += 1
            continue

        ready.append(row)

    print(f"\n{name.upper()}")
    print("Original:", len(rows))
    print("Dropped (needs_alignment):", dropped_alignment)
    print("Dropped (implausible wps > %.0f):" % WPS_CEILING, dropped_implausible)
    print("Training-ready:", len(ready))

    return ready

train_ready = filter_manifest(train_rows, "train")
val_ready = filter_manifest(val_rows, "validation")

FILTERED_DIR = MANIFEST_DIR / "filtered_for_training"
FILTERED_DIR.mkdir(exist_ok=True)

for name, rows in [("train", train_ready), ("validation", val_ready)]:
    path = FILTERED_DIR / f"{name}_manifest_filtered.jsonl"
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")
    print("Saved:", path)


TRAIN
Original: 3702
Dropped (needs_alignment): 64
Dropped (implausible wps > 8): 25
Training-ready: 3613

VALIDATION
Original: 903
Dropped (needs_alignment): 18
Dropped (implausible wps > 8): 2
Training-ready: 883
Saved: /content/drive/MyDrive/legacyagent/manifests/filtered_for_training/train_manifest_filtered.jsonl
Saved: /content/drive/MyDrive/legacyagent/manifests/filtered_for_training/validation_manifest_filtered.jsonl


In [7]:
# DAY 5 - CELL 6
# Load processor and 4-bit quantized Qwen3-ASR
import torch
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen3ASRForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.bfloat16,
)

print("Model loaded:", type(model).__name__)
print("GPU allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

Model loaded: Qwen3ASRForConditionalGeneration
GPU allocated: 2.25 GB


In [8]:
# Check EOS token and chat template (informational)
print("EOS token:", processor.tokenizer.eos_token)
print("EOS token id:", processor.tokenizer.eos_token_id)

sample_msg = [{"role": "assistant", "content": "test transcript"}]
sample_prompt = processor.apply_chat_template(sample_msg, tokenize=False)
print("Chat template sample:", repr(sample_prompt))

EOS token: <|im_end|>
EOS token id: 151645
Chat template sample: '<|im_start|>system\n<|im_end|>\n<|im_start|>user\n<|im_end|>\n<|im_start|>assistant\ntest transcript<|im_end|>\n'


In [9]:
# DAY 5 - CELL 7
# Generate LoRA target modules: all language_model attention
# projections, all layers - excludes audio_tower explicitly by
# requiring the "model.language_model." prefix.

ATTENTION_PROJECTIONS = {"q_proj", "k_proj", "v_proj", "o_proj"}

target_modules = []

for name, module in model.named_modules():
    class_name = type(module).__name__
    is_linear = isinstance(module, torch.nn.Linear) or "Linear4bit" in class_name

    if not is_linear:
        continue

    if not name.startswith("model.language_model."):
        continue

    leaf = name.split(".")[-1]

    if leaf in ATTENTION_PROJECTIONS:
        target_modules.append(name)

print("Total LoRA target modules:", len(target_modules))
print("First 8:", target_modules[:8])

assert all("audio_tower" not in name for name in target_modules)
print("No audio_tower modules included - OK")

Total LoRA target modules: 112
First 8: ['model.language_model.layers.0.self_attn.q_proj', 'model.language_model.layers.0.self_attn.k_proj', 'model.language_model.layers.0.self_attn.v_proj', 'model.language_model.layers.0.self_attn.o_proj', 'model.language_model.layers.1.self_attn.q_proj', 'model.language_model.layers.1.self_attn.k_proj', 'model.language_model.layers.1.self_attn.v_proj', 'model.language_model.layers.1.self_attn.o_proj']
No audio_tower modules included - OK


In [10]:
# DAY 5 - CELL 8
# Prepare model for k-bit training and attach full LoRA config
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)

print("LoRA attached")
model.print_trainable_parameters()

LoRA attached
trainable params: 12,845,056 || all params: 2,050,897,536 || trainable%: 0.6263


In [11]:
# DAY 5 - CELL 9
# Dataset + collator (EOS-fixed)
#
# Labels are built manually: tokenize the prompt (with audio
# placeholder) alone to get its length, then mask that prefix
# with -100 so loss is computed only on the transcript tokens.
# The transcript is followed by the EOS/end-of-turn token so the
# model is trained to predict a stop signal right after it.

import librosa
from torch.utils.data import Dataset

class LegacyAgentASRDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


def build_training_example(row, processor):
    audio, _ = librosa.load(
        row["audio_path"], sr=16000,
        offset=row["start"], duration=row["duration"]
    )

    messages = [{"role": "user", "content": [{"type": "audio", "audio": audio}]}]

    prompt_text = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )

    eos_text = processor.tokenizer.eos_token

    prompt_inputs = processor(text=prompt_text, audio=audio, return_tensors="pt")
    prompt_len = prompt_inputs["input_ids"].shape[1]

    full_inputs = processor(
        text=prompt_text + row["text"] + eos_text,
        audio=audio,
        output_labels=True,
        return_tensors="pt"
    )

    labels = full_inputs["input_ids"].clone()
    labels[:, :prompt_len] = -100
    full_inputs["labels"] = labels

    return full_inputs

def collate_fn(batch):
    row = batch[0]
    example = build_training_example(row, processor)
    return {k: v for k, v in example.items() if isinstance(v, torch.Tensor)}


train_dataset = LegacyAgentASRDataset(train_ready)
val_dataset = LegacyAgentASRDataset(val_ready)

print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

Train examples: 3613
Validation examples: 883


In [12]:
# SANITY CHECK - confirm the EOS fix actually labels the token
test_example = build_training_example(train_ready[0], processor)
last_label = test_example["labels"][0, -1].item()
print("Last label token id:", last_label)
print("Expected EOS id:", processor.tokenizer.eos_token_id)
print("Decoded last token:", processor.tokenizer.decode([last_label]))
assert last_label == processor.tokenizer.eos_token_id, "EOS fix did not take - stop and investigate"
print("EOS token correctly present in labels - OK")

Last label token id: 151645
Expected EOS id: 151645
Decoded last token: <|im_end|>
EOS token correctly present in labels - OK


In [13]:
# DAY 5 - CELL 11
# TrainingArguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=20,
    save_strategy="steps",
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_total_limit=5,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

print("Training args set")

Training args set


In [14]:
# CHECK - confirm what is on disk in the EOS-fix run folder
print("Exists:", EOSFIX_CHECKPOINT_DIR.exists())
if EOSFIX_CHECKPOINT_DIR.exists():
    for p in sorted(EOSFIX_CHECKPOINT_DIR.iterdir()):
        print(" -", p.name)

Exists: True
 - checkpoint-100
 - checkpoint-150
 - checkpoint-200
 - checkpoint-250
 - checkpoint-50
 - continued_from_250
 - final_adapter


In [15]:
CONTINUED_DIR = EOSFIX_CHECKPOINT_DIR / "continued_from_250"
print("Exists:", CONTINUED_DIR.exists())
if CONTINUED_DIR.exists():
    for p in sorted(CONTINUED_DIR.iterdir()):
        print(" -", p.name)

Exists: True
 - checkpoint-250
 - checkpoint-300
 - checkpoint-350
 - checkpoint-400
 - checkpoint-450


In [16]:
from peft import PeftModel
from transformers import Trainer, EarlyStoppingCallback
import os

CONTINUED_DIR = EOSFIX_CHECKPOINT_DIR / "continued_from_250"

# Find the latest checkpoint from THIS continuation run (not the old checkpoint-250)
existing = [d for d in os.listdir(CONTINUED_DIR) if d.startswith("checkpoint-")] if CONTINUED_DIR.exists() else []

if existing:
    latest = sorted(existing, key=lambda x: int(x.split("-")[1]))[-1]
    RESUME_FROM = str(CONTINUED_DIR / latest)
    print("Resuming from this run's own checkpoint:", RESUME_FROM)
else:
    RESUME_FROM = str(EOSFIX_CHECKPOINT_DIR / "checkpoint-250")
    print("No progress found yet, starting from checkpoint-250:", RESUME_FROM)

model = PeftModel.from_pretrained(model.get_base_model(), RESUME_FROM)

for name, param in model.named_parameters():
    if "lora" in name.lower():
        param.requires_grad = True

model.print_trainable_parameters()
model.train()

training_args.output_dir = str(CONTINUED_DIR)
training_args.max_steps = 600
training_args.num_train_epochs = 1

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

# THIS TIME resume_from_checkpoint works because it's resuming its OWN
# consistent optimizer state (created under the fix), not the old broken one
trainer.train(resume_from_checkpoint=RESUME_FROM if existing else None)

Resuming from this run's own checkpoint: /content/drive/MyDrive/legacyagent/checkpoints/qlora_v1_run2_eosfix/continued_from_250/checkpoint-450


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 12,845,056 || all params: 2,050,897,536 || trainable%: 0.6263


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
500,0.189278,0.608929
550,0.195273,0.627265
600,0.191109,0.630583


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=600, training_loss=0.048299649953842165, metrics={'train_runtime': 3783.2729, 'train_samples_per_second': 1.269, 'train_steps_per_second': 0.159, 'total_flos': 9731236096430592.0, 'train_loss': 0.048299649953842165, 'epoch': 1.3277055078881816})

In [17]:
# DAY 5 - CELL 13
# Save the EOS-fixed adapter
FINAL_ADAPTER_DIR_V2 = EOSFIX_CHECKPOINT_DIR / "final_adapter"
model.save_pretrained(str(FINAL_ADAPTER_DIR_V2))
processor.save_pretrained(str(FINAL_ADAPTER_DIR_V2))
print("EOS-fixed adapter saved to:", FINAL_ADAPTER_DIR_V2)

EOS-fixed adapter saved to: /content/drive/MyDrive/legacyagent/checkpoints/qlora_v1_run2_eosfix/final_adapter
